# BKwG — worked examples

**The beta Kumaraswamy-G family of distributions in R.**

This notebook runs on an **R** runtime. In Colab choose *Runtime → Change runtime type → R*.

Authors: Mohamad Yamen AL Mohamad, Ali Akbar Heydari, Hossein Jabbari Khamnei
(Department of Statistics, University of Tabriz).

Contents:
1. Installation
2. The family, baselines and sub-models
3. Shapes of the density and the hazard
4. Distribution functions
5. Structural properties
6. Simulation and maximum likelihood estimation
7. Six estimation methods
8. Sub-models: likelihood ratio and Wald tests
9. Goodness of fit with bootstrap p-values
10. A real data set: model comparison
11. A user-defined baseline
12. A small Monte Carlo study

## 1. Installation

In [ ]:
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
if (!requireNamespace("numDeriv", quietly = TRUE)) install.packages("numDeriv")
remotes::install_github("yamenetoo/Statistical-Repository", subdir = "BKwG")
library(BKwG)
options(repr.plot.width = 12, repr.plot.height = 6, digits = 5)

## 2. The family, baselines and sub-models

For a baseline cdf $G$ with pdf $g$, let $K(x) = 1 - [1 - G(x)^a]^b$ (Kumaraswamy-G). The BKw-G distribution has

$$F(x) = I_{K(x)}(\alpha, \beta), \qquad f(x) = \frac{a\,b\,g\,G^{a-1}(1-G^a)^{b\beta-1}K^{\alpha-1}}{B(\alpha,\beta)}.$$

The baseline is chosen with `baseline =` and its parameters with `par =`.

In [ ]:
bkwg_baselines()
bkwg_submodels()

## 3. Shapes of the density and the hazard

The four shape parameters give very different densities from the same Weibull baseline.

In [ ]:
par(mfrow = c(1, 2))
cols <- c("black", "red", "blue", "darkgreen", "purple")
sets <- list(c(1, 1, 1, 1), c(2, 1, 1.5, 2), c(0.5, 3, 2, 0.5),
             c(3, 0.5, 0.4, 2), c(1.5, 2, 4, 0.3))
plot_bkwg(1, 1, 1, 1, par = c(1.5, 1), from = 0.01, to = 3, ylim = c(0, 2.2),
          main = "BKw-Weibull densities", lwd = 2)
for (k in 2:5) { s <- sets[[k]]
  plot_bkwg(s[1], s[2], s[3], s[4], par = c(1.5, 1), from = 0.01, to = 3,
            add = TRUE, col = cols[k], lwd = 2) }
legend("topright", legend = sapply(sets, paste, collapse = ", "),
       col = cols, lwd = 2, title = "(a, b, alpha, beta)", bty = "n")

## hazard shapes: increasing, decreasing, bathtub, upside-down bathtub
plot_bkwg(2, 1, 1, 1, par = c(1.2, 1), what = "hazard", from = 0.01, to = 3,
          ylim = c(0, 4), lwd = 2, main = "Hazard shapes")
plot_bkwg(0.5, 1, 0.6, 1, par = c(1, 1), what = "hazard", from = 0.01, to = 3,
          add = TRUE, col = "red", lwd = 2)
plot_bkwg(0.3, 0.6, 1, 1, par = c(2.5, 1), what = "hazard", from = 0.01,
          to = 1.6, add = TRUE, col = "blue", lwd = 2)
plot_bkwg(4, 1, 1, 1, par = c(1, 1), baseline = "lnorm", what = "hazard",
          from = 0.01, to = 3, add = TRUE, col = "darkgreen", lwd = 2)
par(mfrow = c(1, 1))

## 4. Distribution functions

Density, cdf, quantile, random generation, survival, hazard and cumulative hazard. All tails are computed on the log scale.

In [ ]:
x <- c(0.5, 1, 2)
dbkwg(x, a = 2, b = 0.7, alpha = 1.6, beta = 2.3, baseline = "weibull",
      par = c(shape = 1.5, scale = 2))
pbkwg(x, 2, 0.7, 1.6, 2.3, par = c(1.5, 2))
qbkwg(c(0.1, 0.5, 0.9), 2, 0.7, 1.6, 2.3, par = c(1.5, 2))
sbkwg(x, 2, 0.7, 1.6, 2.3, par = c(1.5, 2))
hbkwg(x, 2, 0.7, 1.6, 2.3, par = c(1.5, 2))
chbkwg(x, 2, 0.7, 1.6, 2.3, par = c(1.5, 2))

## checks: the density integrates to one, Q inverts F, extreme tails stay accurate
integrate(dbkwg, 0, Inf, a = 2, b = 0.7, alpha = 1.6, beta = 2.3, par = c(1.5, 2))
u <- c(1e-8, 0.3, 0.999999)
pbkwg(qbkwg(u, 2, 0.7, 1.6, 2.3, par = c(1.5, 2)), 2, 0.7, 1.6, 2.3, par = c(1.5, 2))
pbkwg(40, 2, 0.7, 1.6, 2.3, par = c(1.5, 2), lower.tail = FALSE, log.p = TRUE)

In [ ]:
## any other baseline works the same way
dbkwg(c(-1, 0, 1), 2, 1, 0.5, 2, baseline = "norm", par = c(mean = 0, sd = 1))
pbkwg(c(1, 5), 1.5, 2, 1, 3, baseline = "lomax", par = c(shape = 2, scale = 1))
pbkwg(c(0.2, 0.7), 2, 3, 1.5, 2, baseline = "unif")   # beta-Kumaraswamy on (0,1)

## the special cases reduce to known distributions
xx <- c(0.5, 1.5)
all.equal(dbkwg(xx, par = c(2, 3)), dweibull(xx, 2, 3))                 # G
G <- pweibull(xx, 1.5)
all.equal(pbkwg(xx, 2, 3, 1, 1, par = c(1.5, 1)), 1 - (1 - G^2)^3)     # Kw-G
all.equal(pbkwg(xx, 1, 1, 2.5, 0.7, par = c(1.5, 1)), pbeta(G, 2.5, 0.7)) # Beta-G

## 5. Structural properties

In [ ]:
th <- list(a = 2, b = 1, alpha = 1.5, beta = 2, par = c(2, 1))
mbkwg(1:4, th$a, th$b, th$alpha, th$beta, par = th$par)            # raw moments
bkwg_stats(th$a, th$b, th$alpha, th$beta, par = th$par)            # summary measures
mgf_bkwg(c(-1, 0.5, 1), th$a, th$b, th$alpha, th$beta, par = th$par)
entropy_bkwg(th$a, th$b, th$alpha, th$beta, par = th$par)          # Shannon
entropy_bkwg(th$a, th$b, th$alpha, th$beta, par = th$par, type = "renyi", delta = 0.5)
mrl_bkwg(c(0.25, 0.5, 1), th$a, th$b, th$alpha, th$beta, par = th$par)

In [ ]:
## skewness and kurtosis as functions of alpha
al <- seq(0.3, 5, length.out = 25)
sk <- t(sapply(al, function(v) bkwg_stats(2, 1, v, 2, par = c(2, 1))[c("skewness", "kurtosis")]))
par(mfrow = c(1, 2))
plot(al, sk[, 1], type = "l", lwd = 2, xlab = expression(alpha), ylab = "skewness")
plot(al, sk[, 2], type = "l", lwd = 2, xlab = expression(alpha), ylab = "kurtosis")
par(mfrow = c(1, 1))

In [ ]:
## order statistics: densities of the minimum and maximum of N = 10
xg <- seq(0.01, 2.5, length.out = 300)
plot(xg, dbkwg(xg, 2, 1, 1.5, 2, par = c(2, 1)), type = "l", lwd = 2,
     ylim = c(0, 4.5), xlab = "x", ylab = "density", main = "Order statistics, N = 10")
lines(xg, dos_bkwg(xg, 1, 10, 2, 1, 1.5, 2, par = c(2, 1)), col = "blue", lwd = 2)
lines(xg, dos_bkwg(xg, 10, 10, 2, 1, 1.5, 2, par = c(2, 1)), col = "red", lwd = 2)
legend("topright", c("f(x)", "minimum", "maximum"), col = c("black", "blue", "red"),
       lwd = 2, bty = "n")

## mixture representation f = sum_j w_j * KwG(a, b(beta + j))
w <- bkwg_mixture(alpha = 2.5, beta = 1.5, J = 10)
head(w); sum(bkwg_mixture(2.5, 1.5, J = 2000)$weight)
cbind(series = dbkwg_series(c(0.5, 1, 2), 2, 1, 2.5, 1.5, par = c(2, 1), J = 500),
      exact  = dbkwg(c(0.5, 1, 2), 2, 1, 2.5, 1.5, par = c(2, 1)))

## 6. Simulation and maximum likelihood estimation

In [ ]:
set.seed(123)
x <- rbkwg(400, a = 2, b = 0.8, alpha = 0.7, beta = 1.5, baseline = "weibull",
           par = c(shape = 1.5, scale = 2))
fit <- fit_bkwg(x, baseline = "weibull")
summary(fit)
confint(fit, type = "log")
coef(fit, complete = TRUE)
c(AIC = AIC(fit), BIC = BIC(fit))

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 9)
plot(fit)
options(repr.plot.width = 12, repr.plot.height = 6)

## analytical score at the estimate (close to zero for the shape parameters)
e <- coef(fit, complete = TRUE)
score_bkwg(x, e["a"], e["b"], e["alpha"], e["beta"], par = e[c("shape", "scale")])

## 7. Six estimation methods

Maximum likelihood (`mle`), maximum product of spacings (`mps`), minimum Anderson–Darling (`ad`), minimum Cramér–von Mises (`cvm`), least squares (`lse`) and weighted least squares (`wlse`). Here the Kw-Weibull sub-model with known scale.

In [ ]:
set.seed(7)
y <- rbkwg(250, a = 2, b = 1.5, baseline = "weibull", par = c(shape = 1.2, scale = 1))
methods <- c("mle", "mps", "ad", "cvm", "lse", "wlse")
est <- t(sapply(methods, function(m)
  coef(fit_bkwg(y, baseline = "weibull", submodel = "KwG", method = m,
                fixed = c(scale = 1)))))
round(est, 4)      # true values: a = 2, b = 1.5, shape = 1.2

## 8. Sub-models: likelihood ratio and Wald tests

In [ ]:
set.seed(3)
z <- rbkwg(400, a = 3, b = 0.5, alpha = 2, beta = 0.6, par = c(1, 1))
fz <- fit_bkwg(z, baseline = "weibull", fixed = c(scale = 1))
lrt_bkwg(fz, "KwG")      # H0: alpha = beta = 1
lrt_bkwg(fz, "BetaG")    # H0: a = b = 1
lrt_bkwg(fz, "G")        # H0: plain Weibull
wald_bkwg(fz, c(alpha = 1, beta = 1))

## 9. Goodness of fit with bootstrap p-values

`W*` and `A*` (Chen & Balakrishnan, 1995) and Kolmogorov–Smirnov. With `B > 0` the model is re-estimated on each bootstrap sample.

In [ ]:
fk <- fit_bkwg(z, baseline = "weibull", submodel = "KwG", fixed = c(scale = 1))
gof_bkwg(fk)
gof_bkwg(fk, B = 100, seed = 1)

## 10. A real data set: model comparison

Annual precipitation (inches) of 70 US cities (`datasets::precip`). We compare sub-models and baselines with AIC, CAIC, BIC, HQIC, W*, A* and KS.

In [ ]:
data(precip)
summary(precip)
tab <- compare_bkwg(precip, baseline = c("weibull", "gamma", "lnorm"),
                    submodels = c("BKwG", "KwG", "BetaG", "EKwG", "G"))
print(tab, digits = 4)

In [ ]:
best <- attr(tab, "fits")[[tab$model[1]]]
summary(best)
options(repr.plot.width = 12, repr.plot.height = 9)
plot(best)
options(repr.plot.width = 12, repr.plot.height = 6)

## 11. A user-defined baseline

Any continuous distribution can be plugged in. Here a Rayleigh baseline; the quantile function is optional.

In [ ]:
rayleigh <- bkwg_baseline("rayleigh",
  pG = function(q, par) ifelse(q > 0, 1 - exp(-q^2 / (2 * par[["sigma"]]^2)), 0),
  dG = function(x, par) ifelse(x > 0, x / par[["sigma"]]^2 *
                               exp(-x^2 / (2 * par[["sigma"]]^2)), 0),
  qG = function(p, par) par[["sigma"]] * sqrt(-2 * log1p(-p)),
  par = c(sigma = 1), lower = 0, upper = Inf, support = c(0, Inf),
  start = function(x) c(sigma = sqrt(mean(x^2) / 2)))
rayleigh

set.seed(11)
r <- rbkwg(300, a = 2, b = 1, alpha = 1.5, beta = 0.8, baseline = rayleigh,
           par = c(sigma = 2))
fr <- fit_bkwg(r, baseline = rayleigh, submodel = "EKwG")
summary(fr)

## 12. A small Monte Carlo study

Bias and root mean squared error of the MLE in the exponentiated Kumaraswamy-exponential model (`EKwG`, rate fixed at 1) for increasing sample sizes. Increase `R` for publication-quality results.

In [ ]:
set.seed(2026)
true <- c(a = 2, b = 1.5, alpha = 0.8)
R <- 100
res <- lapply(c(50, 100, 300), function(n) {
  est <- t(replicate(R, {
    s <- rbkwg(n, true[["a"]], true[["b"]], true[["alpha"]], 1,
               baseline = "exp", par = c(rate = 1))
    coef(fit_bkwg(s, baseline = "exp", submodel = "EKwG", fixed = c(rate = 1),
                  hessian = FALSE, nstart = 2))
  }))
  data.frame(n = n, parameter = names(true),
             bias = colMeans(est) - true,
             rmse = sqrt(colMeans(sweep(est, 2, true)^2)))
})
do.call(rbind, res)

---
**Citation.** Run `citation("BKwG")`. The family was introduced by Handique, L., Chakraborty, S. and Ali, M. M. (2017), *Beta generated Kumaraswamy-G family of distributions*, Pakistan Journal of Statistics 33(6), 467–490.